# Fraud Detection — Stage 2 CatBoost + XGBoost Ensemble

## Clean end-to-end ML pipeline

This notebook is the **Stage 2 ensemble** built on the project's existing feature-engineering pipeline.

### Pipeline
1. Load the already-engineered train/validation datasets.
2. Prepare CatBoost and XGBoost inputs.
3. Train CatBoost and XGBoost on the same temporal train/validation split.
4. Evaluate ROC-AUC and PR-AUC.
5. Select the CatBoost/XGBoost blend using validation PR-AUC.
6. Tune the classification threshold on validation F1.
7. Report Precision, Recall, F1 and the confusion matrix.
8. Reconstruct the **exact feature-engineering pipeline used in `lightgbm(4).ipynb`** for the real Kaggle test data using **train-derived mappings only**.
9. Verify `engineered_test.csv` has exactly the model feature contract.
10. Refit final CatBoost/XGBoost models on train + validation using the selected model settings.
11. Generate final blended test probabilities and save `ensemble_submission.csv`.

> **Important:** The real Kaggle test has no `isFraud`, so Precision/Recall/F1 cannot be calculated on it. Those metrics are evaluated on the labeled validation set. The final test output is a fraud probability/risk score.

## 1. Imports and configuration

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

DATA_DIR = Path(r"C:\Users\madhu\OneDrive\Desktop\cts")
MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

TARGET = "isFraud"
RANDOM_STATE = 42

TRAIN_PATH = DATA_DIR / "engineered_train.csv"
VAL_PATH = DATA_DIR / "engineered_validation.csv"

RAW_TRAIN_TRANSACTION = DATA_DIR / "train_transaction.csv"
RAW_TRAIN_IDENTITY = DATA_DIR / "train_identity.csv"
RAW_TEST_TRANSACTION = DATA_DIR / "test_transaction.csv"
RAW_TEST_IDENTITY = DATA_DIR / "test_identity.csv"

ENGINEERED_TEST_PATH = DATA_DIR / "engineered_test.csv"
SUBMISSION_PATH = DATA_DIR / "ensemble_submission.csv"

print("Project directory:", DATA_DIR)


Project directory: C:\Users\madhu\OneDrive\Desktop\cts


## 2. Load the existing engineered train/validation data

These files were already produced by the original feature-engineering notebook. We do **not** recreate train/validation features here.

In [2]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Missing: {TRAIN_PATH}")
if not VAL_PATH.exists():
    raise FileNotFoundError(f"Missing: {VAL_PATH}")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Train columns:", len(train_df.columns))
print("Validation columns:", len(val_df.columns))

assert list(train_df.columns) == list(val_df.columns), (
    "Train and validation feature contracts do not match."
)
assert TARGET in train_df.columns and TARGET in val_df.columns


Train shape: (472432, 84)
Validation shape: (118108, 84)
Train columns: 84
Validation columns: 84


## 3. Prepare model matrices

In [3]:
X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].astype(int).copy()

X_val = val_df.drop(columns=[TARGET]).copy()
y_val = val_df[TARGET].astype(int).copy()

DROP_FEATURES = [c for c in ["TransactionID", "uid"] if c in X_train.columns]

X_train = X_train.drop(columns=DROP_FEATURES)
X_val = X_val.drop(columns=DROP_FEATURES)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("Fraud rate:", y_train.mean())
print("Negative/positive ratio:", (y_train == 0).sum() / (y_train == 1).sum())


X_train: (472432, 81)
X_val  : (118108, 81)
Fraud rate: 0.03513521522674162
Negative/positive ratio: 27.46147358274595


## 4. Define categorical features

These are the categorical columns used by the existing Stage 2 notebook. Missing values are represented explicitly for CatBoost.

In [4]:
CAT_COLS = [
    "ProductCD",
    "card4",
    "card6",
    "DeviceType",
    "M1", "M2", "M3",
    "M4", "M5", "M6",
    "M7", "M8", "M9",
    "P_emaildomain_bin",
    "R_emaildomain_bin",
    "DeviceInfo_bin",
]
CAT_COLS = [c for c in CAT_COLS if c in X_train.columns]

for col in CAT_COLS:
    X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
    X_val[col] = X_val[col].fillna("__MISSING__").astype(str)

remaining = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
if set(remaining) != set(CAT_COLS):
    raise ValueError(
        f"Unexpected string columns. Expected {CAT_COLS}; found {remaining}"
    )

print("Categorical columns:", CAT_COLS)


Categorical columns: ['ProductCD', 'card4', 'card6', 'DeviceType', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P_emaildomain_bin', 'R_emaildomain_bin', 'DeviceInfo_bin']


## 5. Train CatBoost

In [5]:
cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    od_type="Iter",
    od_wait=100,
    verbose=100,
    thread_count=-1,
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=CAT_COLS,
    eval_set=(X_val, y_val),
)

cat_best_iteration = cat_model.get_best_iteration()
print("CatBoost best iteration:", cat_best_iteration)

cat_model.save_model(MODEL_DIR / "catboost_stage2_validation.cbm")


0:	test: 0.8210845	best: 0.8210845 (0)	total: 2.36s	remaining: 58m 54s
100:	test: 0.8792374	best: 0.8792374 (100)	total: 3m 16s	remaining: 45m 21s
200:	test: 0.8902142	best: 0.8902142 (200)	total: 8m 33s	remaining: 55m 16s
300:	test: 0.8987876	best: 0.8988957 (296)	total: 11m 41s	remaining: 46m 35s
400:	test: 0.9052202	best: 0.9052202 (400)	total: 14m 53s	remaining: 40m 47s
500:	test: 0.9086364	best: 0.9090312 (483)	total: 18m 5s	remaining: 36m 5s
600:	test: 0.9083261	best: 0.9091776 (511)	total: 21m 16s	remaining: 31m 49s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.909177637
bestIteration = 511

Shrink model to first 512 iterations.
CatBoost best iteration: 511


## 6. CatBoost validation performance

In [6]:
cat_val_pred = cat_model.predict_proba(X_val)[:, 1]

cat_roc_auc = roc_auc_score(y_val, cat_val_pred)
cat_pr_auc = average_precision_score(y_val, cat_val_pred)

print(f"CatBoost ROC-AUC: {cat_roc_auc:.6f}")
print(f"CatBoost PR-AUC : {cat_pr_auc:.6f}")


CatBoost ROC-AUC: 0.909178
CatBoost PR-AUC : 0.508679


## 7. Prepare XGBoost categorical data

In [7]:
# Use one fixed category definition for train and validation.
for col in CAT_COLS:
    categories = pd.concat(
        [X_train[col], X_val[col]],
        ignore_index=True
    ).astype("category").cat.categories

    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_train[col] = X_train[col].astype(dtype)
    X_val[col] = X_val[col].astype(dtype)

xgb_cat_cols = X_train.select_dtypes(include=["category"]).columns.tolist()

print("XGBoost categorical columns:", xgb_cat_cols)
print(
    "Remaining object/string columns:",
    X_train.select_dtypes(include=["object", "string"]).columns.tolist()
)


XGBoost categorical columns: ['ProductCD', 'card4', 'card6', 'DeviceType', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P_emaildomain_bin', 'R_emaildomain_bin', 'DeviceInfo_bin']
Remaining object/string columns: []


## 8. Train XGBoost

In [8]:
xgb_model = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    reg_lambda=5,
    random_state=RANDOM_STATE,
    early_stopping_rounds=100,
    n_jobs=-1,
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

xgb_best_iteration = int(xgb_model.best_iteration)
print("XGBoost best iteration:", xgb_best_iteration)

xgb_model.save_model(MODEL_DIR / "xgboost_stage2_validation.json")


[0]	validation_0-auc:0.80821
[100]	validation_0-auc:0.90186
[200]	validation_0-auc:0.91007
[300]	validation_0-auc:0.91387
[400]	validation_0-auc:0.91646
[500]	validation_0-auc:0.91870
[600]	validation_0-auc:0.91993
[700]	validation_0-auc:0.91989
[734]	validation_0-auc:0.91962
XGBoost best iteration: 634


## 9. XGBoost validation performance

In [9]:
xgb_val_pred = xgb_model.predict_proba(X_val)[:, 1]

xgb_roc_auc = roc_auc_score(y_val, xgb_val_pred)
xgb_pr_auc = average_precision_score(y_val, xgb_val_pred)

print(f"XGBoost ROC-AUC: {xgb_roc_auc:.6f}")
print(f"XGBoost PR-AUC : {xgb_pr_auc:.6f}")


XGBoost ROC-AUC: 0.920126
XGBoost PR-AUC : 0.564536


## 10. Select CatBoost/XGBoost blend

PR-AUC is the primary blend-selection metric because fraud is highly imbalanced.

In [10]:
weights = np.arange(0.0, 1.01, 0.05)
blend_results = []

for cat_weight in weights:
    xgb_weight = 1.0 - cat_weight
    blended = cat_weight * cat_val_pred + xgb_weight * xgb_val_pred

    blend_results.append({
        "catboost_weight": cat_weight,
        "xgboost_weight": xgb_weight,
        "roc_auc": roc_auc_score(y_val, blended),
        "pr_auc": average_precision_score(y_val, blended),
    })

blend_results = pd.DataFrame(blend_results)
best_blend_row = blend_results.loc[blend_results["pr_auc"].idxmax()]

BEST_CAT_WEIGHT = float(best_blend_row["catboost_weight"])
BEST_XGB_WEIGHT = float(best_blend_row["xgboost_weight"])

print(blend_results.sort_values("pr_auc", ascending=False).head(10).to_string(index=False))
print(f"\nSelected CatBoost weight: {BEST_CAT_WEIGHT:.2f}")
print(f"Selected XGBoost weight : {BEST_XGB_WEIGHT:.2f}")


 catboost_weight  xgboost_weight  roc_auc   pr_auc
            0.05            0.95 0.922778 0.565081
            0.00            1.00 0.920126 0.564536
            0.10            0.90 0.922166 0.563856
            0.15            0.85 0.921179 0.562194
            0.20            0.80 0.920156 0.560333
            0.25            0.75 0.919177 0.558388
            0.30            0.70 0.918267 0.556355
            0.35            0.65 0.917419 0.554232
            0.40            0.60 0.916635 0.552063
            0.45            0.55 0.915901 0.549804

Selected CatBoost weight: 0.05
Selected XGBoost weight : 0.95


## 11. Evaluate the selected validation blend

In [11]:
blend_val_pred = (
    BEST_CAT_WEIGHT * cat_val_pred
    + BEST_XGB_WEIGHT * xgb_val_pred
)

blend_roc_auc = roc_auc_score(y_val, blend_val_pred)
blend_pr_auc = average_precision_score(y_val, blend_val_pred)

print(f"Blend ROC-AUC: {blend_roc_auc:.6f}")
print(f"Blend PR-AUC : {blend_pr_auc:.6f}")


Blend ROC-AUC: 0.922778
Blend PR-AUC : 0.565081


## 12. Tune classification threshold on validation

The threshold is selected on the labeled validation set only. It is **not** re-tuned using the real Kaggle test set.

In [12]:
fine_thresholds = np.arange(0.05, 0.501, 0.01)
threshold_results = []

for threshold in fine_thresholds:
    pred = (blend_val_pred >= threshold).astype(int)
    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_results)
BEST_THRESHOLD = float(
    threshold_results.loc[threshold_results["f1"].idxmax(), "threshold"]
)

print("Top thresholds by F1:")
print(threshold_results.sort_values("f1", ascending=False).head(10).to_string(index=False))
print(f"\nSelected validation threshold: {BEST_THRESHOLD:.2f}")


Top thresholds by F1:
 threshold  precision   recall       f1
      0.23   0.598160 0.512057 0.551770
      0.26   0.628859 0.491142 0.551534
      0.25   0.618494 0.497047 0.551160
      0.24   0.606995 0.503937 0.550686
      0.27   0.636599 0.484498 0.550231
      0.28   0.645806 0.477362 0.548953
      0.22   0.581517 0.518701 0.548316
      0.29   0.656121 0.469488 0.547332
      0.21   0.566182 0.527313 0.546057
      0.30   0.665018 0.463091 0.545982

Selected validation threshold: 0.23


## 13. Final validation metrics

In [13]:
blend_val_class = (blend_val_pred >= BEST_THRESHOLD).astype(int)

print("FINAL VALIDATION METRICS")
print("========================")
print(f"ROC-AUC  : {blend_roc_auc:.6f}")
print(f"PR-AUC   : {blend_pr_auc:.6f}")
print(f"Precision: {precision_score(y_val, blend_val_class, zero_division=0):.6f}")
print(f"Recall   : {recall_score(y_val, blend_val_class, zero_division=0):.6f}")
print(f"F1       : {f1_score(y_val, blend_val_class, zero_division=0):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, blend_val_class))

print("\nClassification Report:")
print(classification_report(y_val, blend_val_class, digits=4))


FINAL VALIDATION METRICS
ROC-AUC  : 0.922778
PR-AUC   : 0.565081
Precision: 0.598160
Recall   : 0.512057
F1       : 0.551770

Confusion Matrix:
[[112646   1398]
 [  1983   2081]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9827    0.9877    0.9852    114044
           1     0.5982    0.5121    0.5518      4064

    accuracy                         0.9714    118108
   macro avg     0.7904    0.7499    0.7685    118108
weighted avg     0.9695    0.9714    0.9703    118108



## 14. Save validation outputs and selected settings

In [14]:
np.save(MODEL_DIR / "catboost_val_pred.npy", cat_val_pred)
np.save(MODEL_DIR / "xgboost_val_pred.npy", xgb_val_pred)
np.save(MODEL_DIR / "ensemble_val_pred.npy", blend_val_pred)

settings = {
    "catboost_weight": BEST_CAT_WEIGHT,
    "xgboost_weight": BEST_XGB_WEIGHT,
    "classification_threshold": BEST_THRESHOLD,
    "catboost_best_iteration": int(cat_best_iteration),
    "xgboost_best_iteration": int(xgb_best_iteration),
    "catboost_roc_auc": cat_roc_auc,
    "catboost_pr_auc": cat_pr_auc,
    "xgboost_roc_auc": xgb_roc_auc,
    "xgboost_pr_auc": xgb_pr_auc,
    "ensemble_roc_auc": blend_roc_auc,
    "ensemble_pr_auc": blend_pr_auc,
}

with open(MODEL_DIR / "ensemble_settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print(json.dumps(settings, indent=2))


{
  "catboost_weight": 0.05,
  "xgboost_weight": 0.95,
  "classification_threshold": 0.23,
  "catboost_best_iteration": 511,
  "xgboost_best_iteration": 634,
  "catboost_roc_auc": 0.9091776369570855,
  "catboost_pr_auc": 0.5086785935461408,
  "xgboost_roc_auc": 0.9201264066093291,
  "xgboost_pr_auc": 0.5645359752770974,
  "ensemble_roc_auc": 0.9227778494872956,
  "ensemble_pr_auc": 0.5650808988553311
}


## 15. Build `engineered_test.csv` using the original feature-engineering pipeline

The real Kaggle test is currently raw:

- `test_transaction.csv`
- `test_identity.csv`

The code below reproduces the feature transformations from the original `lightgbm(4).ipynb`.

**Data-leakage rule:** every learned mapping/statistic below is fitted from the original training portion only and then applied to test:
- top email/device categories
- UID mean/std
- UID count
- frequency encoding
- V-column representative selection
- categorical code definitions

The order of operations intentionally matches the original LightGBM notebook: sparse columns are identified first, missingness flags are created from them, and the original sparse columns are dropped only afterward.

No target from test is used.

In [15]:
# Check required raw files
for p in [
    RAW_TRAIN_TRANSACTION,
    RAW_TRAIN_IDENTITY,
    RAW_TEST_TRANSACTION,
    RAW_TEST_IDENTITY,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing raw file: {p}")

print("All raw train/test files found.")


All raw train/test files found.


In [16]:
# Load raw train and raw test
raw_train_transaction = pd.read_csv(RAW_TRAIN_TRANSACTION)
raw_train_identity = pd.read_csv(RAW_TRAIN_IDENTITY)

raw_test_transaction = pd.read_csv(RAW_TEST_TRANSACTION)
raw_test_identity = pd.read_csv(RAW_TEST_IDENTITY)

raw_train = raw_train_transaction.merge(
    raw_train_identity,
    on="TransactionID",
    how="left",
)

raw_test = raw_test_transaction.merge(
    raw_test_identity,
    on="TransactionID",
    how="left",
)

print("Raw train:", raw_train.shape)
print("Raw test :", raw_test.shape)


Raw train: (590540, 434)
Raw test : (506691, 433)


In [17]:
# Reproduce the original temporal split used to learn feature-engineering statistics.
raw_train = raw_train.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(raw_train) * 0.80)
fe_train = raw_train.iloc[:split_idx].copy()
fe_val = raw_train.iloc[split_idx:].copy()
fe_test = raw_test.copy()

print("Feature-engineering source train:", fe_train.shape)
print("Feature-engineering source val  :", fe_val.shape)
print("Feature-engineering target test :", fe_test.shape)


Feature-engineering source train: (472432, 434)
Feature-engineering source val  : (118108, 434)
Feature-engineering target test : (506691, 433)


In [18]:
# Reproduce the original near-constant and >95% sparse-column filtering.
# IMPORTANT: the original pipeline calculates sparse columns here but does NOT
# drop them until after missingness flags and engineered features are created.

nunique = fe_train.nunique(dropna=False)
near_constant_cols = nunique[nunique <= 1].index.tolist()
near_constant_cols = [
    c for c in near_constant_cols
    if c not in [TARGET, "TransactionID"]
]

# Near-constant columns are removed immediately, exactly as in lightgbm(4).ipynb.
for frame in (fe_train, fe_val, fe_test):
    frame.drop(
        columns=[c for c in near_constant_cols if c in frame.columns],
        inplace=True,
        errors="ignore",
    )

missing_frac = fe_train.isna().mean()
very_sparse_cols = missing_frac[missing_frac > 0.95].index.tolist()
very_sparse_cols = [
    c for c in very_sparse_cols
    if c not in [TARGET, "TransactionID"]
]

# Do NOT drop very_sparse_cols yet. The original pipeline first creates
# missingness indicators such as id_02_isna from these columns, and only
# later removes the original sparse columns.

print("Near-constant removed:", len(near_constant_cols))
print("Very sparse columns marked:", len(very_sparse_cols))


Near-constant removed: 0
Very sparse columns marked: 9


In [20]:
# ============================================================
# MISSINGNESS FLAGS
# ============================================================

flag_candidates = [
    "id_02",
    "id_05",
    "id_06",
    "id_11",
    "DeviceInfo",
]

# Only create flags for columns that exist in TRAIN.
# If a column is completely absent from TEST, treat it as
# completely missing in TEST.
flag_candidates = [
    c for c in flag_candidates
    if c in fe_train.columns
]

for col in flag_candidates:

    # -------------------------
    # TRAIN
    # -------------------------
    if col in fe_train.columns:
        fe_train[f"{col}_isna"] = (
            fe_train[col].isna().astype(int)
        )
    else:
        raise ValueError(
            f"Required source column '{col}' is missing from TRAIN."
        )

    # -------------------------
    # VALIDATION
    # -------------------------
    if col in fe_val.columns:
        fe_val[f"{col}_isna"] = (
            fe_val[col].isna().astype(int)
        )
    else:
        raise ValueError(
            f"Required source column '{col}' is missing from VALIDATION."
        )

    # -------------------------
    # TEST
    # -------------------------
    if col in fe_test.columns:
        fe_test[f"{col}_isna"] = (
            fe_test[col].isna().astype(int)
        )
    else:
        # Column does not exist at all in TEST.
        # Therefore every TEST row is considered missing.
        fe_test[f"{col}_isna"] = 1


# ============================================================
# LOG TRANSFORMATION OF TRANSACTION AMOUNT
# ============================================================

if "TransactionAmt" in fe_train.columns:

    if "TransactionAmt" not in fe_val.columns:
        raise ValueError(
            "TransactionAmt is missing from VALIDATION."
        )

    if "TransactionAmt" not in fe_test.columns:
        raise ValueError(
            "TransactionAmt is missing from TEST."
        )

    fe_train["TransactionAmt_log"] = np.log1p(
        fe_train["TransactionAmt"]
    )

    fe_val["TransactionAmt_log"] = np.log1p(
        fe_val["TransactionAmt"]
    )

    fe_test["TransactionAmt_log"] = np.log1p(
        fe_test["TransactionAmt"]
    )


print("Missingness flags created:")
print([
    f"{col}_isna"
    for col in flag_candidates
])


Missingness flags created:
['id_02_isna', 'id_05_isna', 'id_06_isna', 'id_11_isna', 'DeviceInfo_isna']


In [21]:
# Confirm that sparse source columns can exist long enough to create their flags.
# The original raw columns may be >95% missing, but their *_isna features must remain.
created_flag_cols = [c for c in fe_train.columns if c.endswith("_isna")]
print("Created missingness flags:", created_flag_cols)


Created missingness flags: ['id_02_isna', 'id_05_isna', 'id_06_isna', 'id_11_isna', 'DeviceInfo_isna']


In [22]:
# Top-category bins — mappings learned from feature-engineering TRAIN only.
def fit_top_categories(series, top_n=15):
    return series.value_counts(dropna=False).head(top_n).index

def apply_top_categories(series, top_values, other_label="other"):
    return series.where(series.isin(top_values), other_label)

top_category_maps = {}

for col, top_n in [
    ("P_emaildomain", 15),
    ("R_emaildomain", 15),
    ("DeviceInfo", 20),
]:
    if col in fe_train.columns:
        top_values = fit_top_categories(fe_train[col], top_n=top_n)
        top_category_maps[col] = top_values

        output_col = f"{col}_bin"
        for frame in (fe_train, fe_val, fe_test):
            frame[output_col] = apply_top_categories(frame[col], top_values)

print("Top-category mappings created for:", list(top_category_maps))


Top-category mappings created for: ['P_emaildomain', 'R_emaildomain', 'DeviceInfo']


In [23]:
# UID — exactly card1 + addr1 + D1.
uid_required = ["card1", "addr1", "D1"]
missing_uid_cols = [c for c in uid_required if c not in fe_train.columns]

if missing_uid_cols:
    raise ValueError(f"Cannot create UID; missing columns: {missing_uid_cols}")

def create_uid(frame):
    frame["uid"] = (
        frame["card1"].astype(str)
        + "_"
        + frame["addr1"].astype(str)
        + "_"
        + frame["D1"].astype(str)
    )
    return frame

for frame in (fe_train, fe_val, fe_test):
    create_uid(frame)

print("Train UID count:", fe_train["uid"].nunique())
print("Test UID count :", fe_test["uid"].nunique())


Train UID count: 192316
Test UID count : 205299


In [24]:
# Time features — exact transformations from the original notebook.
def create_time_features(frame):
    frame["TransactionDay"] = frame["TransactionDT"] // (24 * 60 * 60)
    frame["D1n"] = frame["TransactionDay"] - frame["D1"]
    frame["Transaction_hour"] = (frame["TransactionDT"] // 3600) % 24
    frame["Transaction_dayofweek"] = (frame["TransactionDT"] // (3600 * 24)) % 7
    return frame

for frame in (fe_train, fe_val, fe_test):
    create_time_features(frame)


In [25]:
# UID aggregate statistics — fitted ONLY on feature-engineering TRAIN.
agg_cols = [
    "TransactionAmt",
    "D4",
    "D10",
    "D15",
    "C1",
    "C13",
]
agg_cols = [c for c in agg_cols if c in fe_train.columns]

uid_agg_maps = {}

for col in agg_cols:
    uid_mean = fe_train.groupby("uid")[col].mean()
    uid_std = fe_train.groupby("uid")[col].std()

    uid_agg_maps[col] = (uid_mean, uid_std)

    fe_train[f"{col}_uid_mean"] = fe_train["uid"].map(uid_mean)
    fe_train[f"{col}_uid_std"] = fe_train["uid"].map(uid_std)

    fe_val[f"{col}_uid_mean"] = fe_val["uid"].map(uid_mean)
    fe_val[f"{col}_uid_std"] = fe_val["uid"].map(uid_std)

    fe_test[f"{col}_uid_mean"] = fe_test["uid"].map(uid_mean)
    fe_test[f"{col}_uid_std"] = fe_test["uid"].map(uid_std)

# UID count — also train-derived.
uid_count_map = fe_train["uid"].value_counts()

for frame in (fe_train, fe_val, fe_test):
    frame["uid_count"] = frame["uid"].map(uid_count_map).fillna(0)

print("UID aggregate columns:", agg_cols)


UID aggregate columns: ['TransactionAmt', 'D4', 'D10', 'D15', 'C1', 'C13']


In [26]:
# Frequency encoding — fitted ONLY on feature-engineering TRAIN.
freq_encode_cols = [
    "card1",
    "card2",
    "addr1",
    "P_emaildomain",
    "DeviceInfo",
]
freq_encode_cols = [c for c in freq_encode_cols if c in fe_train.columns]

freq_maps = {}

for col in freq_encode_cols:
    freq_map = fe_train[col].value_counts(normalize=True)
    freq_maps[col] = freq_map

    for frame in (fe_train, fe_val, fe_test):
        frame[f"{col}_freq"] = frame[col].map(freq_map).fillna(0)

print("Frequency-encoded columns:", freq_encode_cols)


Frequency-encoded columns: ['card1', 'card2', 'addr1', 'P_emaildomain', 'DeviceInfo']


In [27]:
# V-column representative selection — target-informed selection is fitted on TRAIN only.
v_cols = [
    c for c in fe_train.columns
    if c.startswith("V") and c[1:].isdigit()
]

def select_v_representatives(data, v_cols, target):
    if not v_cols:
        return []

    missing_rate = data[v_cols].isna().mean().round(4)
    groups = {}

    for col, rate in missing_rate.items():
        groups.setdefault(rate, []).append(col)

    representatives = []

    for _, cols in groups.items():
        if len(cols) == 1:
            representatives.append(cols[0])
            continue

        correlations = data[cols].corrwith(data[target]).abs().dropna()

        if len(correlations) == 0:
            representatives.append(cols[0])
        else:
            representatives.append(correlations.idxmax())

    return representatives

v_representatives = select_v_representatives(
    fe_train,
    v_cols,
    TARGET,
)

v_cols_to_drop = [c for c in v_cols if c not in v_representatives]

# Match the original pipeline: remove very sparse columns and non-representative
# V columns only after missingness flags and all engineered features are created.
drop_cols = list(dict.fromkeys(very_sparse_cols + v_cols_to_drop))
drop_cols = [c for c in drop_cols if c not in [TARGET, "TransactionID"]]

for frame in (fe_train, fe_val, fe_test):
    frame.drop(
        columns=[c for c in drop_cols if c in frame.columns],
        inplace=True,
        errors="ignore",
    )

print("V columns originally:", len(v_cols))
print("V representatives    :", len(v_representatives))
print("Columns removed after engineering:", len(drop_cols))


V columns originally: 339
V representatives    : 14
Columns removed after engineering: 334


In [28]:
# Reproduce the exact final feature contract from the original pipeline.
keep_as_is = [
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "D1",
    "D4",
    "D10",
    "D15",
    "DeviceType",
]

keep_as_is += [
    c for c in fe_train.columns
    if c.startswith("C") and c[1:].isdigit()
]

keep_as_is += [
    c for c in fe_train.columns
    if c.startswith("M") and c[1:].isdigit()
]

keep_as_is = list(dict.fromkeys(
    c for c in keep_as_is if c in fe_train.columns
))

engineered_features = [
    "uid",
    "D1n",
    "uid_count",
    "Transaction_hour",
    "Transaction_dayofweek",
    "TransactionAmt_log",
    "P_emaildomain_bin",
    "R_emaildomain_bin",
    "DeviceInfo_bin",
]

for col in agg_cols:
    engineered_features += [
        f"{col}_uid_mean",
        f"{col}_uid_std",
    ]

for col in freq_encode_cols:
    engineered_features.append(f"{col}_freq")

for col in flag_candidates:
    engineered_features.append(f"{col}_isna")

engineered_features = list(dict.fromkeys(
    c for c in engineered_features if c in fe_train.columns
))

# Match the exact feature list represented by the existing engineered_train.csv.
existing_train_columns = pd.read_csv(
    TRAIN_PATH,
    nrows=0
).columns.tolist()

final_features = [
    c for c in existing_train_columns
    if c not in [TARGET, "TransactionID"]
]

# Safety check: every expected feature must exist in engineered test source.
missing_test_features = [
    c for c in final_features
    if c not in fe_test.columns
]

if missing_test_features:
    raise ValueError(
        "Test engineering did not create these model features:\n"
        + "\n".join(missing_test_features)
    )

print("Existing model feature count:", len(final_features))
print("All model features exist in engineered test source.")


Existing model feature count: 82
All model features exist in engineered test source.


In [29]:
# Apply the same TRAIN-derived categorical code definitions used in lightgbm(4).ipynb.
# This is important: test categories are never fitted independently.

categorical_cols = [
    c for c in fe_train.select_dtypes(include=["object", "category"]).columns
    if c in (keep_as_is + engineered_features)
]

for col in categorical_cols:
    train_categories = fe_train[col].dropna().unique()

    dtype = pd.api.types.CategoricalDtype(categories=train_categories)

    for frame in (fe_train, fe_val, fe_test):
        frame[col] = frame[col].astype(dtype).cat.codes.replace(-1, np.nan)

print("Categorical columns encoded:", len(categorical_cols))


Categorical columns encoded: 17


In [30]:
# Create the actual engineered test file.
engineered_test = fe_test[["TransactionID"] + final_features].copy()

# Match the column order of engineered_train exactly, apart from isFraud.
expected_test_columns = [
    c for c in pd.read_csv(TRAIN_PATH, nrows=0).columns
    if c != TARGET
]

if list(engineered_test.columns) != expected_test_columns:
    raise ValueError(
        "Final engineered test columns do not exactly match the train contract."
    )

engineered_test.to_csv(ENGINEERED_TEST_PATH, index=False)

print("Saved:", ENGINEERED_TEST_PATH)
print("Engineered test shape:", engineered_test.shape)
print("Expected feature count:", len(final_features))


Saved: C:\Users\madhu\OneDrive\Desktop\cts\engineered_test.csv
Engineered test shape: (506691, 83)
Expected feature count: 82


## 16. Verify the engineered test contract

In [31]:
# Reload exactly as the ensemble model will consume it.
test_check = pd.read_csv(ENGINEERED_TEST_PATH)
train_columns = pd.read_csv(TRAIN_PATH, nrows=0).columns.tolist()
test_columns = test_check.columns.tolist()

print("Train columns:", len(train_columns))
print("Test columns :", len(test_columns))

missing_from_test = [c for c in train_columns if c != TARGET and c not in test_columns]
extra_in_test = [c for c in test_columns if c not in train_columns]

print("\nMissing from test:", missing_from_test)
print("Extra in test   :", extra_in_test)

assert missing_from_test == []
assert extra_in_test == []
assert "isFraud" not in test_check.columns

print("\nTEST FEATURE CONTRACT: PASS")


Train columns: 84
Test columns : 83

Missing from test: []
Extra in test   : []

TEST FEATURE CONTRACT: PASS


## 17. Prepare final train + validation data for refitting

Model selection is already finished. We now lock the selected blend weights and threshold, then refit each model on **all labeled data** using the best validation-derived number of boosting iterations.

In [32]:
# Reload the original engineered datasets to avoid any category mutations from validation training.
final_train_df = pd.read_csv(TRAIN_PATH)
final_val_df = pd.read_csv(VAL_PATH)

full_labeled = pd.concat(
    [final_train_df, final_val_df],
    ignore_index=True,
)

X_full = full_labeled.drop(columns=[TARGET]).copy()
y_full = full_labeled[TARGET].astype(int).copy()

X_full = X_full.drop(
    columns=[c for c in ["TransactionID", "uid"] if c in X_full.columns]
)

X_final_test = test_check.drop(
    columns=[c for c in ["TransactionID", "uid"] if c in test_check.columns]
).copy()

test_ids = test_check["TransactionID"].copy()

for col in CAT_COLS:
    X_full[col] = X_full[col].fillna("__MISSING__").astype(str)
    X_final_test[col] = X_final_test[col].fillna("__MISSING__").astype(str)

print("Full labeled data:", X_full.shape)
print("Final test data  :", X_final_test.shape)


Full labeled data: (590540, 81)
Final test data  : (506691, 81)


## 18. Prepare final XGBoost category definitions

In [33]:
for col in CAT_COLS:
    categories = pd.concat(
        [X_full[col], X_final_test[col]],
        ignore_index=True
    ).astype("category").cat.categories

    dtype = pd.api.types.CategoricalDtype(categories=categories)

    X_full[col] = X_full[col].astype(dtype)
    X_final_test[col] = X_final_test[col].astype(dtype)

print(
    "Remaining object/string columns:",
    X_full.select_dtypes(include=["object", "string"]).columns.tolist()
)


Remaining object/string columns: []


## 19. Train final CatBoost on all labeled data

In [34]:
final_cat_iterations = max(int(cat_best_iteration) + 1, 1)

final_cat_model = CatBoostClassifier(
    iterations=final_cat_iterations,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=100,
    thread_count=-1,
)

final_cat_model.fit(
    X_full,
    y_full,
    cat_features=CAT_COLS,
)

final_cat_model.save_model(MODEL_DIR / "catboost_stage2_final.cbm")
print("Final CatBoost iterations:", final_cat_iterations)


0:	learn: 0.6663452	total: 2.12s	remaining: 18m 5s
100:	learn: 0.3772117	total: 2m 55s	remaining: 11m 55s
200:	learn: 0.3356635	total: 5m 45s	remaining: 8m 53s
300:	learn: 0.3059416	total: 8m 32s	remaining: 5m 59s
400:	learn: 0.2734820	total: 11m 24s	remaining: 3m 9s
500:	learn: 0.2494988	total: 14m 15s	remaining: 18.8s
511:	learn: 0.2471554	total: 14m 34s	remaining: 0us
Final CatBoost iterations: 512


## 20. Train final XGBoost on all labeled data

In [35]:
final_xgb_iterations = max(int(xgb_best_iteration) + 1, 1)

final_xgb_model = XGBClassifier(
    n_estimators=final_xgb_iterations,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    reg_lambda=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_xgb_model.fit(
    X_full,
    y_full,
    verbose=False,
)

final_xgb_model.save_model(MODEL_DIR / "xgboost_stage2_final.json")
print("Final XGBoost iterations:", final_xgb_iterations)


Final XGBoost iterations: 635


## 21. Generate final test probabilities and blend

In [36]:
xgb_test_pred = final_xgb_model.predict_proba(X_final_test)[:, 1]

cat_test_input = X_final_test.copy()
for col in CAT_COLS:
    cat_test_input[col] = cat_test_input[col].astype(str)

cat_test_pred = final_cat_model.predict_proba(cat_test_input)[:, 1]

blend_test_pred = (
    BEST_XGB_WEIGHT * xgb_test_pred
    + BEST_CAT_WEIGHT * cat_test_pred
)

print("XGBoost test probabilities:", xgb_test_pred.shape)
print("CatBoost test probabilities:", cat_test_pred.shape)
print("Blended test probabilities:", blend_test_pred.shape)

print("\nProbability summary:")
print(pd.Series(blend_test_pred).describe())


XGBoost test probabilities: (506691,)
CatBoost test probabilities: (506691,)
Blended test probabilities: (506691,)

Probability summary:
count    506691.000000
mean          0.041974
std           0.105180
min           0.000081
25%           0.007224
50%           0.016002
75%           0.034889
max           0.995102
dtype: float64


## 22. Create final submission

The submission contains the **continuous fraud probability**, not the thresholded 0/1 class. The selected threshold remains available for operational classification, but ranking/probability is what should be submitted for a probability-based fraud competition.

In [37]:
submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": blend_test_pred,
})

submission.to_csv(SUBMISSION_PATH, index=False)

print("Submission saved:", SUBMISSION_PATH)
print("Submission shape:", submission.shape)
print("\nSubmission preview:")
print(submission.head())
print("\nProbability range:",
      submission["isFraud"].min(),
      "to",
      submission["isFraud"].max())


Submission saved: C:\Users\madhu\OneDrive\Desktop\cts\ensemble_submission.csv
Submission shape: (506691, 2)

Submission preview:
   TransactionID   isFraud
0        3663549  0.001373
1        3663550  0.008951
2        3663551  0.017240
3        3663552  0.010987
4        3663553  0.002272

Probability range: 8.062414333201078e-05 to 0.9951024081471394


## 23. Final project summary

### Validation model-selection results
- CatBoost and XGBoost are evaluated independently.
- The blend is selected using validation PR-AUC.
- The fraud classification threshold is selected using validation F1.
- Precision, Recall, F1 and the confusion matrix are reported on validation.

### Final test
- Raw `test_transaction.csv` + `test_identity.csv` are merged.
- The original feature-engineering pipeline is reproduced.
- Every learned feature mapping/statistic comes from the training portion only.
- Test columns are verified against the existing engineered training feature contract.
- Final CatBoost and XGBoost models are refit on all labeled train + validation data.
- Test probabilities are blended using the validation-selected weights.
- `ensemble_submission.csv` contains the final continuous fraud probabilities.

### Important
The real test set has no labels, so do not calculate or invent test Precision/Recall/F1. Those metrics belong to the labeled validation/local test evaluation.

In [38]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_model(name, y_true, prob, threshold=0.25):

    pred = (prob >= threshold).astype(int)

    print(f"\n{name}")
    print("-" * 40)
    print(f"ROC-AUC   : {roc_auc_score(y_true, prob):.6f}")
    print(f"PR-AUC    : {average_precision_score(y_true, prob):.6f}")
    print(f"Precision : {precision_score(y_true, pred):.6f}")
    print(f"Recall    : {recall_score(y_true, pred):.6f}")
    print(f"F1        : {f1_score(y_true, pred):.6f}")


evaluate_model("XGBoost", y_val, xgb_val_pred)
evaluate_model("CatBoost", y_val, cat_val_pred)
evaluate_model("Ensemble", y_val, blend_val_pred)


XGBoost
----------------------------------------
ROC-AUC   : 0.920126
PR-AUC    : 0.564536
Precision : 0.642528
Recall    : 0.480315
F1        : 0.549704

CatBoost
----------------------------------------
ROC-AUC   : 0.909178
PR-AUC    : 0.508679
Precision : 0.117684
Recall    : 0.870817
F1        : 0.207347

Ensemble
----------------------------------------
ROC-AUC   : 0.922778
PR-AUC    : 0.565081
Precision : 0.618494
Recall    : 0.497047
F1        : 0.551160
